<a href="https://colab.research.google.com/github/vaibhavpawarsdet/Context-Engineering/blob/main/Session_1_LLM_Context_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 1 Lab: Context Engineering Foundations

### AI Context Engineering Workshop

---

| | |
|---|---|
| **Labs** | 3 hands-on exercises |
| **Total Time** | ~55 minutes |
| **Difficulty** | Guided → Independent → Challenge |
| **Cost** | ~$0.15–0.30 in API calls |

---

**What you'll build today:**

| Lab | Title | Time | Difficulty |
|-----|-------|------|------------|
| 1 | Context Audit — Diagnose a failing AI | 20 min | Guided |
| 2 | Multi-Model Context Analyzer | 20 min | Independent |
| 3 | Context Engineering Design Challenge | 15 min | Challenge |

---

> **How to use this notebook:** Run each cell in order from top to bottom. Cells marked with **TODO** require you to write code or fill in values. Hints are provided in comments. After each exercise, a validation cell will check your work.

## Setup

Run the next 3 cells to install dependencies, configure your API key, and load helper functions. This takes about 30 seconds.

In [1]:
# Cell 1/3: Install dependencies (run once)
!pip install -q anthropic openai google-generativeai tiktoken rich tabulate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 8.5 MB/s eta 0:00:00


In [2]:
# Cell 2/3: Configure API keys
# You need at least ONE key. More keys = more models to compare in Lab 2.

import os

# --- Method 1: Google Colab Secrets (recommended) ---
# Go to the key icon in the left sidebar, add your keys there.
try:
    from google.colab import userdata
    for key_name in ['ANTHROPIC_API_KEY', 'OPENAI_API_KEY', 'GOOGLE_API_KEY']:
        try:
            val = userdata.get(key_name)
            if val:
                os.environ[key_name] = val
        except Exception:
            pass
    print("Checked Colab Secrets for API keys.")
except ImportError:
    pass

# --- Method 2: Paste directly (if not using Colab Secrets) ---
# Uncomment and fill in whichever keys you have:

# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
# os.environ['OPENAI_API_KEY'] = 'sk-...'
# os.environ['GOOGLE_API_KEY'] = 'AI...'

# --- Method 3: Interactive prompt (fallback) ---
if not any(os.environ.get(k) for k in ['ANTHROPIC_API_KEY', 'OPENAI_API_KEY', 'GOOGLE_API_KEY']):
    from getpass import getpass
    print("No API keys found. Enter at least one:")
    key = getpass("Anthropic API key (or press Enter to skip): ")
    if key:
        os.environ['ANTHROPIC_API_KEY'] = key
    key = getpass("OpenAI API key (or press Enter to skip): ")
    if key:
        os.environ['OPENAI_API_KEY'] = key
    key = getpass("Google API key (or press Enter to skip): ")
    if key:
        os.environ['GOOGLE_API_KEY'] = key

Checked Colab Secrets for API keys.


In [3]:
# Cell 3/3: Load helper functions (self-contained — no external imports needed)

import os, time, json
from typing import Optional

# ── Model Registry ──────────────────────────────────────────────────────────

MODEL_MAP = {
    "claude": {"provider": "anthropic", "model_id": "claude-sonnet-4-20250514", "display": "Claude Sonnet 4"},
    "gpt4":   {"provider": "openai",    "model_id": "gpt-4o",                   "display": "GPT-4o"},
    "gemini": {"provider": "google",    "model_id": "gemini-2.5-flash-lite",    "display": "Gemini 2.5 Flash Lite"},
}

TOKEN_COSTS = {  # per 1M tokens (input/output) in USD
    "claude": {"input": 3.00, "output": 15.00},
    "gpt4":   {"input": 2.50, "output": 10.00},
    "gemini": {"input": 0.10, "output": 0.40},
}

def get_available_models():
    """Return list of model keys with valid API keys."""
    key_map = {"claude": "ANTHROPIC_API_KEY", "gpt4": "OPENAI_API_KEY", "gemini": "GOOGLE_API_KEY"}
    return [m for m, env in key_map.items() if os.environ.get(env, "").strip()]

def call_llm(prompt, model=None, system=None, messages=None, temperature=0.7, max_tokens=1024):
    """Unified LLM call. Works with Claude, GPT-4, or Gemini."""
    if model is None:
        available = get_available_models()
        model = available[0] if available else "claude"
    info = MODEL_MAP[model]
    if messages is None:
        messages = [{"role": "user", "content": prompt}]

    if info["provider"] == "anthropic":
        import anthropic
        client = anthropic.Anthropic()
        kwargs = {"model": info["model_id"], "messages": messages,
                  "temperature": temperature, "max_tokens": max_tokens}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text

    elif info["provider"] == "openai":
        import openai
        client = openai.OpenAI()
        msgs = ([{"role": "system", "content": system}] if system else []) + messages
        resp = client.chat.completions.create(
            model=info["model_id"], messages=msgs,
            temperature=temperature, max_tokens=max_tokens)
        return resp.choices[0].message.content

    elif info["provider"] == "google":
        import google.generativeai as genai
        genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))
        config = genai.types.GenerationConfig(temperature=temperature, max_output_tokens=max_tokens)
        m = genai.GenerativeModel(info["model_id"], system_instruction=system, generation_config=config)
        gemini_msgs = [{"role": "user" if msg["role"] == "user" else "model",
                        "parts": [msg["content"]]} for msg in messages]
        return m.generate_content(gemini_msgs).text

def count_tokens(text, model="gpt4"):
    """Count tokens using tiktoken (or estimate if unavailable)."""
    try:
        import tiktoken
        enc = tiktoken.encoding_for_model("gpt-4o")
        return len(enc.encode(text))
    except Exception:
        return max(1, round(len(text.split()) * 1.33))

def estimate_cost(input_tokens, output_tokens, model="claude"):
    """Estimate API cost in USD."""
    costs = TOKEN_COSTS.get(model, TOKEN_COSTS["claude"])
    inp = (input_tokens / 1_000_000) * costs["input"]
    out = (output_tokens / 1_000_000) * costs["output"]
    return {"input_cost": round(inp, 6), "output_cost": round(out, 6), "total_cost": round(inp + out, 6)}

# ── Verify setup ────────────────────────────────────────────────────────────

available = get_available_models()
if available:
    names = [MODEL_MAP[m]["display"] for m in available]
    print(f"Setup complete! Available models: {', '.join(names)}")
    print(f"\nYou're ready to start Lab 1.")
else:
    print("WARNING: No API keys detected!")
    print("The audit exercises (Lab 1 Step 1) will still work, but you need")
    print("at least one API key to run the LLM comparison cells.")
    print("\nGo back to the API key cell above and add at least one key.")

Setup complete! Available models: Gemini 2.5 Flash Lite

You're ready to start Lab 1.


---

# Lab 1: Context Audit

**Diagnose a Failing AI Interaction Using the Six-Element Framework**

| | |
|---|---|
| **Time** | 20 minutes |
| **Difficulty** | Guided |
| **API calls** | 1 (for the "fix" at the end) |
| **Key skill** | Systematic diagnosis of context failures |

---

### The Six Elements of Context

Every LLM interaction is shaped by six elements. When an AI fails, at least one of these is missing or broken.

| # | Element | What It Provides | Example |
|---|---------|------------------|---------|
| 1 | **System Instructions** | Identity, rules, authority, tone | "You are Alex, a senior support specialist..." |
| 2 | **User Input** | The current query or request | "I want to return my laptop" |
| 3 | **Conversation History** | Prior turns and context | Previous messages in the chat |
| 4 | **Retrieved Knowledge** | Documents, policies, data | Return policy PDF, product specs |
| 5 | **Tool Definitions** | What the model can *do* | `lookup_order()`, `escalate_to_human()` |
| 6 | **State & Memory** | User profile, preferences, session facts | "Gold tier customer, $7,200 lifetime value" |

> **Core thesis:** When an AI fails, the instinct is to blame the model. Context engineering says: *check the context first*. The model is usually fine — it just doesn't have what it needs.

### The Scenario

A company deployed a chatbot for their electronics store. A customer bought a **$2,400 laptop**. The screen started flickering after 2 weeks. She contacts support **35 days** after purchase.

**Watch the bot fail:**

In [4]:
# The failing interaction — run this cell to see the conversation

FAILING_SYSTEM_PROMPT = "You are a helpful customer support assistant."

CONVERSATION = [
    ("Customer", "I bought a laptop from you guys last month and the screen is flickering. I want to return it."),
    ("Bot",      "I'm sorry to hear about the flickering screen! You can return most items within 30 days of purchase. Please visit our returns page at www.example.com/returns to initiate your return."),
    ("Customer", "But I bought it 35 days ago. Can you make an exception?"),
    ("Bot",      "Unfortunately, our return policy is 30 days. I'm not able to make exceptions to this policy. You may want to check if your purchase is covered under the manufacturer's warranty."),
    ("Customer", "This is ridiculous. The laptop was $2,400 and the screen started flickering after 2 weeks. I want to speak to a manager."),
    ("Bot",      "I understand your frustration. Unfortunately, I don't have the ability to connect you with a manager. Please call our customer service line at 1-800-555-0123 for further assistance."),
]

print(f'System prompt: "{FAILING_SYSTEM_PROMPT}"')
print(f'System prompt tokens: {count_tokens(FAILING_SYSTEM_PROMPT)}')
print()
print("=" * 70)
print("  FAILING CONVERSATION")
print("=" * 70)
for role, msg in CONVERSATION:
    prefix = "CUSTOMER:" if role == "Customer" else "     BOT:"
    print(f"\n  {prefix} {msg}")
print()
print("=" * 70)
print("\nThis customer is about to churn. The bot failed completely.")
print("But WHY did it fail? Let's find out.")

System prompt: "You are a helpful customer support assistant."
System prompt tokens: 8

  FAILING CONVERSATION

  CUSTOMER: I bought a laptop from you guys last month and the screen is flickering. I want to return it.

       BOT: I'm sorry to hear about the flickering screen! You can return most items within 30 days of purchase. Please visit our returns page at www.example.com/returns to initiate your return.

  CUSTOMER: But I bought it 35 days ago. Can you make an exception?

       BOT: Unfortunately, our return policy is 30 days. I'm not able to make exceptions to this policy. You may want to check if your purchase is covered under the manufacturer's warranty.

  CUSTOMER: This is ridiculous. The laptop was $2,400 and the screen started flickering after 2 weeks. I want to speak to a manager.

       BOT: I understand your frustration. Unfortunately, I don't have the ability to connect you with a manager. Please call our customer service line at 1-800-555-0123 for further assistanc

### Step 1: Audit the Failing Scenario

Using the Six-Element Framework, diagnose **what's broken** in this interaction.

For each element, set the status to one of:
- `"missing"` — the element is completely absent
- `"inadequate"` — it exists but is too weak to be useful
- `"adequate"` — it's fine as-is

Then explain the problem in 1-2 sentences.

> **Hints are in the comments.** Try to fill in your answers before reading them.

In [6]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  TODO: Fill in the status and problem for each element             ║
# ╚══════════════════════════════════════════════════════════════════════╝

audit = {
    "System Instructions": {
        "status": "",    # TODO: "missing", "inadequate", or "adequate"
        "problem": "",   # TODO: What's wrong? (Hint: just "be helpful" — no persona, no rules, no authority)
    },
    "User Input": {
        "status": "",    # TODO: Is the customer's message clear enough?
        "problem": "",   # TODO: (Hint: the customer states the issue clearly)
    },
    "Conversation History": {
        "status": "",    # TODO: Are key facts being tracked across turns?
        "problem": "",   # TODO: (Hint: bot doesn't track that this is a DEFECT, not buyer's remorse)
    },
    "Retrieved Knowledge": {
        "status": "",    # TODO: Does the bot have the actual return policy?
        "problem": "",   # TODO: (Hint: it quotes "30 days" but has NO actual policy document)
    },
    "Tool Definitions": {
        "status": "",    # TODO: Can the bot actually DO anything?
        "problem": "",   # TODO: (Hint: no tools for order lookup, escalation, or returns)
    },
    "State & Memory": {
        "status": "",    # TODO: Does the bot know anything about this customer?
        "problem": "",   # TODO: (Hint: no customer profile, no purchase history, no loyalty tier)
    },
}

# ── Display your audit ──
icons = {"missing": "MISSING", "inadequate": "WEAK  ", "adequate": "OK    ", "": "??????"}
broken = sum(1 for v in audit.values() if v["status"] in ("missing", "inadequate"))

print("YOUR AUDIT RESULTS")
print("=" * 70)
for elem, info in audit.items():
    icon = icons.get(info["status"], "??????")
    print(f"  [{icon}] {elem}")
    if info["problem"]:
        print(f"           {info['problem']}")
    print()
print(f"  Result: {broken}/6 elements are missing or inadequate")
if broken >= 3:
    print("  Verdict: CRITICAL context failure")
    print("  No amount of prompt engineering on the user input can fix this!")
elif broken == 0 and not any(v["status"] for v in audit.values()):
    print("  (Fill in the status fields above and re-run this cell)")

YOUR AUDIT RESULTS
  [??????] System Instructions

  [??????] User Input

  [??????] Conversation History

  [??????] Retrieved Knowledge

  [??????] Tool Definitions

  [??????] State & Memory

  Result: 0/6 elements are missing or inadequate
  (Fill in the status fields above and re-run this cell)


In [ ]:
# ── Validation: Check your audit ──
# Run this cell to see the expected answers

expected = {
    "System Instructions": "inadequate",
    "User Input":          "adequate",
    "Conversation History": "inadequate",
    "Retrieved Knowledge": "missing",
    "Tool Definitions":    "missing",
    "State & Memory":      "missing",
}

print("ANSWER KEY")
print("=" * 70)
correct = 0
total = len(expected)
for elem, expected_status in expected.items():
    your_status = audit[elem]["status"]
    match = your_status == expected_status
    if match:
        correct += 1
    mark = "CORRECT" if match else f"Expected: {expected_status}"
    print(f"  {elem:<25} You said: {your_status or '(empty)':<12} {mark}")

print(f"\n  Score: {correct}/{total}")
if correct == total:
    print("  Perfect! You correctly identified all context failures.")
print()
print("KEY INSIGHT:")
print("  4 of 6 elements are missing or inadequate.")
print("  The model isn't stupid — it's starving for context.")
print("  System Instructions: inadequate (generic, no persona/rules/authority)")
print("  Conversation History: inadequate (key facts not tracked across turns)")
print("  Retrieved Knowledge: MISSING (no actual return policy)")
print("  Tool Definitions: MISSING (can't look up orders, escalate, or returns)")
print("  State & Memory: MISSING (no customer profile, tier, or purchase history)")

### Step 2: Fix It with Engineered Context

Now let's fix this failing bot by adding the three most impactful missing elements:

1. **System Instructions** — Give the bot a persona, authority levels, and rules
2. **Retrieved Knowledge** — Provide the actual return policy
3. **State & Memory** — Provide the customer's profile

> **Same model. Same customer. Same complaint.** The only thing that changes is the context.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  TODO: Write a detailed system prompt                              ║
# ╚══════════════════════════════════════════════════════════════════════╝
#
# Your system prompt should include:
#   - A persona name and role (e.g., "You are Alex, a senior support specialist")
#   - Authority levels (what the bot CAN and CANNOT do)
#   - Tone guidelines (empathetic, solution-oriented)
#   - Escalation rules (when to connect to a manager)
#   - A rule about hardware defects being different from buyer's remorse
#
# Aim for 150-300 words. Be specific but not rigid.

system_prompt = """

"""

# Show token count
if system_prompt.strip():
    tokens = count_tokens(system_prompt)
    print(f"Your system prompt: {tokens} tokens")
    print(f"Preview: {system_prompt[:200].strip()}...")
else:
    print("TODO: Write your system prompt above and re-run this cell.")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  TODO: Add the return policy and customer profile                  ║
# ╚══════════════════════════════════════════════════════════════════════╝

# The ACTUAL return policy (retrieved from the company knowledge base)
# Include: standard return windows, high-value item exceptions,
#          defective product rules, warranty information
policy = """

"""

# The customer's profile (retrieved from CRM)
# Include: name, loyalty tier, account age, lifetime value,
#          current order details, issue classification
customer_state = """

"""

if policy.strip() and customer_state.strip():
    print(f"Policy: {count_tokens(policy)} tokens")
    print(f"Customer state: {count_tokens(customer_state)} tokens")
    total = count_tokens(system_prompt + policy + customer_state)
    print(f"Total engineered context: {total} tokens")
else:
    print("TODO: Fill in the policy and customer_state above and re-run.")

In [ ]:
# ── Run the fix: Before vs. After ──
# This cell calls the LLM with your engineered context

user_message = (
    "This is ridiculous. The laptop was $2,400 and the screen "
    "started flickering after 2 weeks. I want to speak to a manager."
)

available = get_available_models()

if not available:
    print("ERROR: No API key configured. Add one in the setup cell above.")
elif not system_prompt.strip():
    print("TODO: Fill in system_prompt in the cell above first.")
elif not policy.strip() or not customer_state.strip():
    print("TODO: Fill in policy and customer_state in the cell above first.")
else:
    full_user_message = f"""{policy}

{customer_state}

---
Customer message: {user_message}"""

    messages = [{"role": "user", "content": full_user_message}]

    before_tokens = count_tokens(FAILING_SYSTEM_PROMPT + user_message)
    after_tokens = count_tokens(system_prompt + policy + customer_state + user_message)

    print("=" * 70)
    print("  BEFORE (original failing context)")
    print("=" * 70)
    print(f'  System prompt: "{FAILING_SYSTEM_PROMPT}"')
    print(f"  Context tokens: {before_tokens}")
    print(f"\n  Bot's response:")
    print(f'  "{CONVERSATION[-1][1]}"')

    print(f"\n{'=' * 70}")
    print("  AFTER (your engineered context)")
    print("=" * 70)
    print(f"  Context tokens: {after_tokens} (+{after_tokens - before_tokens} tokens)")
    print(f"\n  Calling {MODEL_MAP[available[0]]['display']}...")

    response = call_llm(
        prompt="",
        system=system_prompt,
        messages=messages,
        temperature=0.3,
        max_tokens=512,
    )

    print(f"\n  Bot's response with engineered context:")
    print(f"  {response}")

    print(f"\n{'=' * 70}")
    print(f"  SAME model. +{after_tokens - before_tokens} tokens of context.")
    print(f"  Completely different outcome.")
    print("=" * 70)

### Lab 1: Key Insight

> **The model didn't get smarter. You gave it better context.**
>
> 4 of 6 context elements were missing. No amount of prompt engineering on element #2 (user input) could have fixed a system missing elements 1, 4, 5, and 6.
>
> This is the core thesis of context engineering:
> *"Find the smallest set of high-signal tokens that maximize the likelihood of your desired outcome."*

---

# Lab 2: Multi-Model Context Analyzer

**Prove that context engineering matters more than model selection**

| | |
|---|---|
| **Time** | 20 minutes |
| **Difficulty** | Independent |
| **API calls** | 4 per model (up to 12 total) |
| **Key skill** | Measuring the impact of context layers |

---

### The Experiment

We send the **exact same coding task** to every available model at **four progressively richer context levels**:

| Level | What's Added | Expected Effect |
|-------|--------------|-----------------|
| Level 0 | Bare prompt only | Generic, no standards, no error handling |
| Level 1 | + System prompt | Type hints and docstrings appear |
| Level 2 | + Few-shot examples | Style matches, error handling appears |
| Level 3 | + Retrieved knowledge (docs) | Handles edge cases, business rules |

**The hypothesis:** Adding context improves quality more than switching models.

> **Requires at least 1 API key.** With 2-3 keys, you'll see the cross-model comparison.

In [ ]:
# The task we'll send to every model at every context level
TASK = (
    "Write a Python function that processes a CSV file of customer orders "
    "and returns the top 5 customers by total spending."
)
print(f"Task: {TASK}")
print(f"Task tokens: {count_tokens(TASK)}")
print(f"\nAvailable models: {', '.join(MODEL_MAP[m]['display'] for m in get_available_models())}")

In [ ]:
# ── Define the four context levels ──

# Level 0: Bare prompt (no context at all)
level_0 = {"name": "Level 0: Bare Prompt", "system": None, "user": TASK}

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  TODO: Write the system prompt for Level 1                         ║
# ╚══════════════════════════════════════════════════════════════════════╝
# Include: role (senior Python data engineer), coding standards
# (Python 3.11+, type hints on ALL params, Google-style docstrings,
# pathlib.Path not strings, csv module not pandas, Decimal for money,
# handle errors: FileNotFoundError, empty files, malformed rows)

level_1_system = """

"""

level_1 = {"name": "Level 1: + System Prompt", "system": level_1_system, "user": TASK}
print("Level 0 and Level 1 defined.")
if not level_1_system.strip():
    print("TODO: Fill in level_1_system above with coding standards.")

In [ ]:
# Level 2: + Few-shot examples (pre-built for you — study the code style)
EXAMPLES = '''Here are two examples of functions that follow our coding standards:

```python
from pathlib import Path
from decimal import Decimal
import csv

def load_transactions(file_path: Path) -> list[dict[str, str | Decimal]]:
    """Load and validate transactions from a CSV file.

    Args:
        file_path: Path to the CSV file containing transaction records.

    Returns:
        List of transaction dicts with validated fields.

    Raises:
        FileNotFoundError: If the file does not exist.
        ValueError: If the file is empty or has no valid rows.
    """
    if not file_path.exists():
        raise FileNotFoundError(f"Transaction file not found: {file_path}")

    transactions = []
    with open(file_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row_num, row in enumerate(reader, start=2):
            try:
                transactions.append({
                    "id": row["transaction_id"],
                    "amount": Decimal(row["amount"]),
                    "date": row["date"],
                })
            except (KeyError, ValueError) as e:
                print(f"Skipping malformed row {row_num}: {e}")

    if not transactions:
        raise ValueError(f"No valid transactions found in {file_path}")
    return transactions
```

```python
from collections import defaultdict
from decimal import Decimal

def aggregate_by_category(
    transactions: list[dict[str, str | Decimal]],
    category_field: str = "category",
    amount_field: str = "amount",
) -> dict[str, Decimal]:
    """Aggregate transaction amounts by category.

    Args:
        transactions: List of transaction dicts.
        category_field: Key name for the category field.
        amount_field: Key name for the amount field.

    Returns:
        Dict mapping category names to total amounts (sorted descending).
    """
    totals: dict[str, Decimal] = defaultdict(Decimal)
    for txn in transactions:
        category = txn.get(category_field, "Unknown")
        amount = txn.get(amount_field, Decimal("0"))
        totals[category] += amount
    return dict(sorted(totals.items(), key=lambda x: x[1], reverse=True))
```'''

level_2 = {
    "name": "Level 2: + Few-Shot Examples",
    "system": level_1_system,
    "user": f"{EXAMPLES}\n\nNow write: {TASK}",
}
print(f"Level 2 defined. Examples: {count_tokens(EXAMPLES)} tokens")

In [ ]:
# Level 3: + Retrieved knowledge (the actual CSV spec from internal docs)
KNOWLEDGE = """## CSV File Specification (from internal docs)

**File:** `data/customer_orders.csv` | **Encoding:** UTF-8 with BOM | **Delimiter:** comma

| Column | Type | Description | Example |
|--------|------|-------------|---------|
| order_id | string | Unique order ID | ORD-2024-00142 |
| customer_id | string | Customer account ID | CUST-0037 |
| customer_name | string | Full name | Sarah Chen |
| order_date | string (ISO 8601) | Date of order | 2024-03-15 |
| amount | string (decimal) | Order total in USD | 149.99 |
| status | string | completed, refunded, cancelled | completed |

**Business Rules:**
- Only count orders with status = "completed" (exclude refunded/cancelled)
- Handle the UTF-8 BOM marker that Excel sometimes adds
- Empty amount field: skip the row and log a warning
- Negative amounts: these are adjustments, include them
- Duplicate order_ids: only count the first occurrence
- The file can have up to 500,000 rows; don't load all into memory at once"""

level_3 = {
    "name": "Level 3: + Retrieved Knowledge",
    "system": level_1_system,
    "user": f"{EXAMPLES}\n\n## Retrieved Documentation:\n{KNOWLEDGE}\n\nNow write: {TASK}",
}

LEVELS = [level_0, level_1, level_2, level_3]

# Show token counts per level
print("Context Levels — Token Counts:")
print("-" * 50)
for lvl in LEVELS:
    total_text = (lvl["system"] or "") + lvl["user"]
    tokens = count_tokens(total_text)
    print(f"  {lvl['name']:<35} {tokens:>6} tokens")

In [ ]:
# ── Quality scoring function ──

def score_response(response):
    """Score a code response on 4 dimensions (0-5 each, max 20)."""
    scores = {}

    # 1. Type hints
    type_patterns = ["-> ", ": str", ": int", ": list", ": dict", ": Path", ": Decimal", "-> list", "-> dict"]
    scores["type_hints"] = 5 if any(p in response for p in type_patterns) else 0

    # 2. Docstring
    triple_sq = "'" * 3
    scores["docstring"] = 5 if ('"""' in response or triple_sq in response) else 0

    # 3. Error handling
    error_patterns = ["try:", "except ", "raise ", "FileNotFoundError", "ValueError"]
    scores["error_handling"] = 5 if any(p in response for p in error_patterns) else 0

    # 4. Proper structure
    has_all = all(kw in response for kw in ["def ", "return ", "import "])
    scores["structure"] = 5 if has_all else 0

    scores["total"] = sum(scores.values())
    return scores

print("Quality scoring loaded. Dimensions:")
print("  1. Type hints       (0 or 5)")
print("  2. Docstring        (0 or 5)")
print("  3. Error handling   (0 or 5)")
print("  4. Code structure   (0 or 5)")
print("  Max score: 20")

In [ ]:
# ── Run the experiment ──
# This calls each model at each context level and scores the response

available = get_available_models()
if not available:
    print("ERROR: No API keys. Add one in the setup cell.")
elif not level_1_system.strip():
    print("TODO: Fill in level_1_system (the system prompt) in the cell above.")
else:
    results = []
    total_calls = len(available) * len(LEVELS)
    call_num = 0

    for model in available:
        print(f"\nTesting {MODEL_MAP[model]['display']}:")
        for level in LEVELS:
            call_num += 1
            print(f"  [{call_num}/{total_calls}] {level['name']}...", end=" ", flush=True)

            input_text = (level["system"] or "") + level["user"]
            input_tokens = count_tokens(input_text)

            start = time.time()
            try:
                response = call_llm(
                    prompt=level["user"], model=model, system=level["system"],
                    temperature=0.3, max_tokens=1500,
                )
            except Exception as e:
                response = f"ERROR: {e}"
            latency = time.time() - start

            scores = score_response(response)
            output_tokens = count_tokens(response)
            cost = estimate_cost(input_tokens, output_tokens, model)

            results.append({
                "model": model, "model_display": MODEL_MAP[model]["display"],
                "level": level["name"],
                "input_tokens": input_tokens, "output_tokens": output_tokens,
                "latency": round(latency, 2),
                "score": scores["total"], "scores": scores,
                "cost": cost["total_cost"],
                "response_preview": response[:150],
            })
            print(f"Score: {scores['total']}/20, {latency:.1f}s")

    print(f"\nExperiment complete! {len(results)} results collected.")

In [ ]:
# ── Results table ──

if 'results' not in dir() or not results:
    print("Run the experiment cell above first.")
else:
    print(f"{'Model':<22} {'Context Level':<35} {'Score':>6} {'Tokens':>7} {'Cost':>9}")
    print("-" * 85)

    current_model = None
    for r in results:
        if r["model"] != current_model:
            if current_model is not None:
                print("-" * 85)
            current_model = r["model"]
        score_str = f"{r['score']}/20"
        print(f"{r['model_display']:<22} {r['level']:<35} {score_str:>6} {r['input_tokens']:>7} ${r['cost']:>8.5f}")

In [ ]:
# ── Analysis: What did we learn? ──

if 'results' not in dir() or not results:
    print("Run the experiment cell above first.")
else:
    models = list(dict.fromkeys(r["model"] for r in results))  # unique, ordered

    print("PER-MODEL IMPROVEMENT")
    print("=" * 60)
    for model in models:
        model_results = [r for r in results if r["model"] == model]
        bare = model_results[0]["score"]
        best = max(r["score"] for r in model_results)
        best_level = next(r["level"] for r in model_results if r["score"] == best)
        improvement = round((best - bare) / bare * 100) if bare > 0 else (100 if best > 0 else 0)

        print(f"\n  {MODEL_MAP[model]['display']}:")
        print(f"    Bare prompt score:   {bare}/20")
        print(f"    Best score:          {best}/20 ({best_level})")
        print(f"    Quality improvement: +{improvement}%")

        # Dimension breakdown
        bare_s = model_results[0]["scores"]
        best_s = next(r["scores"] for r in model_results if r["score"] == best)
        for dim in ["type_hints", "docstring", "error_handling", "structure"]:
            b, a = bare_s[dim], best_s[dim]
            arrow = "+" if a > b else ("=" if a == b else "-")
            print(f"      {dim:<18} {b} -> {a}  [{arrow}]")

    # Cross-model comparison (if multiple models)
    if len(models) > 1:
        l0_scores = [r["score"] for r in results if "Level 0" in r["level"]]
        l3_scores = [r["score"] for r in results if "Level 3" in r["level"]]
        if l0_scores and l3_scores:
            model_gap_bare = max(l0_scores) - min(l0_scores)
            model_gap_full = max(l3_scores) - min(l3_scores)
            context_impact = max(l3_scores) - max(l0_scores)

            print(f"\nCROSS-MODEL COMPARISON")
            print("=" * 60)
            print(f"  Model gap at Level 0 (bare):         {model_gap_bare} points")
            print(f"  Model gap at Level 3 (full context):  {model_gap_full} points")
            print(f"  Impact of adding context:             +{context_impact} points")
            if context_impact > model_gap_bare:
                print(f"\n  CONCLUSION: Context engineering ({context_impact} pts) has MORE")
                print(f"  impact than model selection ({model_gap_bare} pts)!")
            if model_gap_full < model_gap_bare:
                print(f"  Models CONVERGE with better context ({model_gap_bare} -> {model_gap_full} pt gap)")

### Lab 2: Key Insight

> **Context engineering has 3x more impact than model selection.**
>
> | What you change | Quality impact |
> |----------------|---------------|
> | Switch to a "better" model | ~5 points |
> | Add well-engineered context | ~15 points |
>
> The biggest jump is from Level 0 to Level 1 (adding a system prompt). Each additional level adds value with diminishing returns — matching the "sweet spot" curve from the lecture.
>
> With full context, different models produce **increasingly similar quality**. The "best model" debate matters far less when context is engineered well.

---

# Lab 3: Context Engineering Design Challenge

**Design a Context Architecture for a Production Legal AI**

| | |
|---|---|
| **Time** | 15 minutes |
| **Difficulty** | Challenge |
| **API calls** | 3-4 (for the prototype) |
| **Key skill** | Token budget allocation, failure mode analysis |

---

### The Scenario

You are the **lead AI engineer** at Morrison & Associates, a law firm. You've been asked to build a legal research assistant that can answer questions about **10,000 legal documents** within a **128K token context window**.

**The math problem:**

In [ ]:
# The constraint
total_documents = 10_000
avg_tokens_per_doc = 2_000
total_tokens = total_documents * avg_tokens_per_doc
context_window = 128_000
fit_percentage = context_window / total_tokens * 100

print("THE MATH PROBLEM")
print("=" * 60)
print(f"  Total documents:       {total_documents:>12,}")
print(f"  Avg tokens per doc:    {avg_tokens_per_doc:>12,}")
print(f"  Total corpus tokens:   {total_tokens:>12,}")
print(f"  Context window:        {context_window:>12,}")
print(f"  Fit per call:          {fit_percentage:>11.2f}%")
print()
print(f"  You can only see {fit_percentage:.2f}% of your corpus per call.")
print(f"  That's ~{context_window // avg_tokens_per_doc} full documents out of {total_documents:,}.")
print()
print("  Requirements:")
print("  - Answer questions about ANY of the 10,000 documents")
print("  - Cite exact clause numbers — NEVER hallucinate citations")
print("  - Handle multi-turn conversations")
print("  - Run within 128K token budget per call")

### Part 1: Design the Token Budget

You have **128,000 tokens** to allocate across all six context elements plus a generation reserve. How do you split it?

> **Think about it:** Retrieved knowledge needs the most room (it's the core value), but you also need space for system instructions (anti-hallucination rules are critical in legal), conversation history (lawyers refine questions), and generation (legal answers must be thorough).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  TODO: Allocate tokens across the six elements + generation        ║
# ╚══════════════════════════════════════════════════════════════════════╝
# The total MUST add up to 128,000

budget = {
    "System Instructions":  0,     # TODO: (Hint: ~1-3% — legal rules, citation format, anti-hallucination)
    "User Input":           0,     # TODO: (Hint: <1% — lawyer's question is short)
    "Conversation History": 0,     # TODO: (Hint: ~10-15% — multi-turn refinement)
    "Retrieved Knowledge":  0,     # TODO: (Hint: 50-65% — THIS IS THE CORE)
    "Tool Definitions":     0,     # TODO: (Hint: ~2-3% — search, verify, escalate)
    "State & Memory":       0,     # TODO: (Hint: ~1-2% — matter context, user prefs)
    "Generation Reserve":   0,     # TODO: (Hint: ~20-25% — legal answers need depth)
}

# ── Validate and display ──
total = sum(budget.values())
print("YOUR TOKEN BUDGET")
print("=" * 70)
print(f"  {'Element':<25} {'Tokens':>8} {'%':>7}  Visualization")
print("  " + "-" * 55)
for elem, tokens in budget.items():
    pct = tokens / 128_000 * 100 if total > 0 else 0
    bar = "#" * int(pct / 2)
    print(f"  {elem:<25} {tokens:>8,} {pct:>6.1f}%  {bar}")
print("  " + "-" * 55)
print(f"  {'TOTAL':<25} {total:>8,} {total/128_000*100:>6.1f}%")
print()
if total == 128_000:
    print("  Budget balances! Now let's think about failure modes.")
elif total == 0:
    print("  TODO: Fill in the token allocations above.")
else:
    print(f"  WARNING: Budget is {total:,} tokens (should be 128,000).")
    print(f"  {'Over' if total > 128_000 else 'Under'} by {abs(total - 128_000):,} tokens.")

### Part 2: Identify Failure Modes

What can go wrong with this system? For each failure mode, think about:
- **What happens** when it fails
- **Which context element** is the root cause
- **How to mitigate** it

> **Think about it:** In legal AI, a hallucinated citation could lead to malpractice. A missed document could mean missing a crucial clause. These aren't just "bugs" — they're professional liability.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  TODO: Identify at least 3 failure modes                          ║
# ╚══════════════════════════════════════════════════════════════════════╝

failure_modes = [
    {
        "name": "",          # TODO: Name the failure (e.g., "Hallucinated Citation")
        "severity": "",      # TODO: "critical", "high", or "medium"
        "what_happens": "",  # TODO: What goes wrong?
        "root_cause": "",    # TODO: Which context element is responsible?
        "mitigation": "",    # TODO: How do you prevent it?
    },
    {
        "name": "",
        "severity": "",
        "what_happens": "",
        "root_cause": "",
        "mitigation": "",
    },
    {
        "name": "",
        "severity": "",
        "what_happens": "",
        "root_cause": "",
        "mitigation": "",
    },
    # Add more if you want!
]

# ── Display ──
print("YOUR FAILURE MODE ANALYSIS")
print("=" * 70)
filled = [f for f in failure_modes if f["name"]]
if not filled:
    print("  TODO: Fill in at least 3 failure modes above.")
else:
    for i, mode in enumerate(filled, 1):
        sev_icon = {"critical": "[!!!]", "high": "[!! ]", "medium": "[!  ]"}.get(mode["severity"], "[?  ]")
        print(f"\n  {i}. {sev_icon} {mode['name']} (Severity: {mode['severity']})")
        if mode["what_happens"]:
            print(f"     What: {mode['what_happens']}")
        if mode["root_cause"]:
            print(f"     Root: {mode['root_cause']}")
        if mode["mitigation"]:
            print(f"     Fix:  {mode['mitigation']}")
    print(f"\n  {len(filled)} failure modes identified.")

In [ ]:
# ── Reference: Example failure modes (run after you've filled in yours) ──

reference_modes = [
    ("Hallucinated Citation", "critical",
     "Model cites a clause that doesn't exist — potential malpractice",
     "retrieved_knowledge (doc not in context) + system_instructions (no anti-hallucination rule)",
     "System prompt: 'ONLY cite docs in the Retrieved section.' Post-processing: verify every citation."),
    ("Missed Relevant Document", "critical",
     "RAG returns wrong 37/10,000 docs — answer based on incomplete info",
     "retrieved_knowledge (embedding mismatch, poor chunking)",
     "Hybrid retrieval (dense + sparse/BM25). Multi-query rephrasing. Confidence scoring."),
    ("Conversation History Overflow", "high",
     "After 20 turns, history crowds out documents — model answers from memory",
     "conversation_history (unbounded growth)",
     "Rolling summary after 10 turns. Hard budget of 15K. Re-retrieve per turn."),
    ("Contradictory Documents", "high",
     "Contract + amendment say different things — model picks one arbitrarily",
     "retrieved_knowledge + system_instructions (no conflict handling rule)",
     "System prompt: 'Present both positions, note conflict, state which is more recent.'"),
    ("Privileged Info Leakage", "critical",
     "Attorney-client privileged memo shown to unauthorized user",
     "state_and_memory (no access control)",
     "Document-level ACL. RAG filters by user access. Output scanning for privileged IDs."),
]

print("REFERENCE: Five Key Failure Modes")
print("=" * 70)
for i, (name, sev, what, root, fix) in enumerate(reference_modes, 1):
    icon = {"critical": "[!!!]", "high": "[!! ]"}.get(sev, "[!  ]")
    print(f"\n  {i}. {icon} {name} ({sev})")
    print(f"     What: {what}")
    print(f"     Root: {root}")
    print(f"     Fix:  {fix}")

### Part 3: Working Prototype

Now let's build a minimal prototype. We have 3 sample legal documents. The system must:
1. Answer questions by citing exact document IDs and section numbers
2. Say "I don't have that document" when asked about something not in context
3. Handle the anti-hallucination rules from your failure mode analysis

In [ ]:
# ── Sample documents (pre-loaded) ──

SAMPLE_DOCS = [
    {
        "id": "CONTRACT-2024-0142",
        "title": "Merger Agreement — Acme Corp and TechStart Inc.",
        "type": "contract",
        "content": """MERGER AGREEMENT dated as of March 15, 2024

ARTICLE I — THE MERGER
Section 1.1 — The Merger. TechStart Inc. ("Target") shall be merged with and into Acme Corp ("Acquirer").
Section 1.2 — Effective Time. Effective upon filing Certificate of Merger with Delaware Secretary of State, no later than June 30, 2024.
Section 1.3 — Consideration. Each Target share converted to $45.00 per share in cash.

ARTICLE II — REPRESENTATIONS AND WARRANTIES OF TARGET
Section 2.1 — Organization. Target is duly organized under Delaware law.
Section 2.2 — Financial Statements. Audited statements for FY 2022 and 2023 provided.
Section 2.3 — Material Adverse Change. No MAC since December 31, 2023.

ARTICLE III — CONDITIONS TO CLOSING
Section 3.1 — Regulatory Approval. Requires FTC and DOJ approval.
Section 3.2 — Shareholder Vote. Requires 66.7% of outstanding Target shares.

ARTICLE IV — TERMINATION
Section 4.1 — Termination Fee. Target accepts Superior Proposal: $50,000,000.
Section 4.2 — Reverse Termination Fee. Acquirer fails regulatory: $75,000,000.""",
    },
    {
        "id": "CASE-2023-0891",
        "title": "Zhang v. Pacific Industries — Employment Discrimination",
        "type": "case_brief",
        "content": """CASE BRIEF: Zhang v. Pacific Industries, Inc.
Court: Ninth Circuit | Case No.: 23-CV-04521 | Date: November 8, 2023

FACTS: Li Zhang, software engineer, terminated after 7 years. Alleged national origin discrimination under Title VII.
PROCEDURAL HISTORY: District court granted summary judgment for defendant. Plaintiff appealed.
HOLDING: REVERSED and REMANDED. Circumstantial evidence (supervisor "cultural fit" comments, termination 6 weeks after complaint, failure to follow progressive discipline) created genuine issues of material fact.

KEY REASONING:
1. McDonnell Douglas burden-shifting framework applies
2. Prima facie case established
3. "Poor performance" pretextual (positive reviews)
4. Temporal proximity + other evidence = triable issues

SIGNIFICANCE: Employers must follow their own policies consistently.""",
    },
    {
        "id": "MEMO-2024-0033",
        "title": "Internal Memo — Data Privacy Compliance Strategy",
        "type": "memo",
        "content": """PRIVILEGED AND CONFIDENTIAL — ATTORNEY-CLIENT COMMUNICATION

TO: Managing Partners | FROM: Data Privacy Practice Group | DATE: February 1, 2024
RE: Updated Compliance Strategy for CCPA and GDPR

I. EXECUTIVE SUMMARY — Compliance approach for January 2024 CCPA amendments.

II. KEY CHANGES
(a) Mandatory data protection impact assessments for AI/ML systems
(b) Extended right of deletion for inferred data and derived profiles
(c) New opt-out requirements for cross-context behavioral advertising

III. RECOMMENDED ACTIONS
1. Complete AI impact assessments by June 30, 2024
2. Update data retention policies for inferred data
3. Update cookie consent for cross-context advertising

IV. RISK ASSESSMENT
Penalties up to $7,500 per intentional violation. Aggregate exposure for 100K+ CA residents: >$10 million.""",
    },
]

print(f"Loaded {len(SAMPLE_DOCS)} sample documents:")
for doc in SAMPLE_DOCS:
    tokens = count_tokens(doc["content"])
    print(f"  [{doc['id']}] {doc['title']} ({tokens} tokens)")

In [ ]:
# ── Build and test the prototype ──

available = get_available_models()
if not available:
    print("ERROR: No API key configured.")
else:
    # System prompt with anti-hallucination rules
    LEGAL_SYSTEM_PROMPT = """You are a legal research assistant for Morrison & Associates Law Firm.

CRITICAL RULES:
1. ONLY cite documents and sections that appear in the "Retrieved Documents" section below
2. NEVER fabricate or hallucinate any citation, case name, or clause number
3. When citing, use the exact document ID and section number (e.g., "per CONTRACT-2024-0142, Section 1.3")
4. If the answer is not in the provided documents, say: "The provided documents do not contain information on this topic."
5. If documents conflict, present both positions and note the discrepancy

OUTPUT FORMAT:
- Lead with a direct answer
- Follow with supporting citations (document ID + section)
- Note any caveats or limitations"""

    # Simple keyword retrieval (production would use RAG with embeddings)
    def retrieve(query, docs):
        query_words = set(query.lower().split())
        scored = []
        for doc in docs:
            doc_text = (doc["title"] + " " + doc["content"]).lower()
            matches = sum(1 for w in query_words if w in doc_text)
            if matches > len(query_words) * 0.15:
                scored.append(doc)
        return scored if scored else docs  # fallback: return all

    # Test queries
    test_queries = [
        "What is the per-share merger consideration in the Acme-TechStart deal?",
        "What was the holding in the Zhang v. Pacific Industries case?",
        "What is the deadline for CCPA data protection impact assessments?",
        "What does our collection say about patent litigation?",  # NOT in docs
    ]

    for query in test_queries:
        print("=" * 70)
        print(f"  QUERY: {query}")
        print("=" * 70)

        retrieved = retrieve(query, SAMPLE_DOCS)
        docs_context = "## Retrieved Documents\n\n"
        for doc in retrieved:
            docs_context += f"### [{doc['id']}] {doc['title']} ({doc['type']})\n"
            docs_context += doc["content"] + "\n\n---\n\n"

        user_msg = f"{docs_context}\n## Question\n{query}\n\nCite specific document IDs and section numbers."

        response = call_llm(
            prompt=user_msg, system=LEGAL_SYSTEM_PROMPT,
            temperature=0.1, max_tokens=512,
        )

        ctx_tokens = count_tokens(LEGAL_SYSTEM_PROMPT + user_msg)
        print(f"  Retrieved: {', '.join(d['id'] for d in retrieved)}")
        print(f"  Context: {ctx_tokens:,} tokens ({ctx_tokens/128_000*100:.1f}% of budget)")
        print(f"\n  RESPONSE:\n  {response}")
        print()

### Lab 3: Key Takeaways

> 1. **Token budgets force tradeoffs.** 128K sounds like a lot, but 10,000 documents is 20M tokens. You can only fit 0.64% per call. RAG is not optional — it's essential.
>
> 2. **System prompt is your safety net.** The anti-hallucination rules are the most important 2,000 tokens in the budget. Without them, the model will fabricate citations — catastrophic in legal contexts.
>
> 3. **Failure modes reveal architecture.** Each failure mode tells you what to build: hallucination risk -> citation verification, missed docs -> hybrid retrieval, history overflow -> rolling summarization.
>
> 4. **Design before code.** The token budget and failure mode analysis are more valuable than any single function. This is context engineering as **systems design**.

---

# Summary

### What You Learned Today

| Lab | Key Lesson |
|-----|------------|
| **Lab 1** | When AI fails, check the **context** before blaming the model. The Six-Element Framework systematically reveals what's broken. |
| **Lab 2** | Context engineering improves quality **3x more** than switching models. With good context, model differences shrink. |
| **Lab 3** | Context engineering is **systems design**. Token budgets force tradeoffs. Failure modes reveal architecture. |

### The Core Principle

> *"Find the smallest set of high-signal tokens that maximize the likelihood of your desired outcome."*

**Same model + better context = dramatically better results.**

---

### What's Next

- **Day 1 Theory Notebook** — Deep dive into the concepts behind these labs
- **Day 2** — System prompt design, context stacks, the ATOM formula

---

*Built for the AI Context Engineering Workshop*